# DGX Spark Notebook 03 - Evaluation (Perplexity) Smoke

Runs a short perplexity evaluation and writes JSON output.
Primary model is Qwen 0.5B; deterministic fallback is microsoft/phi-2 if needed.

In [ ]:
from pathlib import Path
import json
import math
import os
import subprocess

repo_root = Path.cwd()
if not (repo_root / 'scripts').exists() and (repo_root.parent / 'scripts').exists():
    repo_root = repo_root.parent
os.chdir(repo_root)

out_json = repo_root / 'artifacts/notebooks/eval-smoke.json'
out_json.parent.mkdir(parents=True, exist_ok=True)

def run_eval(model_name: str) -> tuple[bool, dict | None]:
    cmd = [
        'python3',
        'scripts/evaluate_perplexity.py',
        '--model', model_name,
        '--quantization', 'fp16',
        '--subset-size', '8',
        '--max-length', '128',
        '--batch-size', '1',
        '--device-map', 'cuda',
        '--output-json', str(out_json),
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.returncode != 0:
        print(proc.stdout[-1500:])
        print(proc.stderr[-1500:])
        return False, None
    payload = json.loads(out_json.read_text())
    result = payload['results'][0]
    ppl = result.get('perplexity')
    if ppl is None:
        return False, result
    if isinstance(ppl, (int, float)) and math.isfinite(float(ppl)):
        return True, result
    print(f'Non-finite perplexity for {model_name}: {ppl}')
    return False, result

candidates = [
    'Qwen/Qwen2.5-0.5B-Instruct',
    'microsoft/phi-2',
]

final_result = None
selected_model = None
for candidate in candidates:
    ok, result = run_eval(candidate)
    if ok:
        final_result = result
        selected_model = candidate
        break

if final_result is None:
    raise RuntimeError('Perplexity evaluation failed for both primary and fallback models')

print('Selected model:', selected_model)
final_result

In [ ]:
payload = json.loads(out_json.read_text())
result = payload['results'][0]
assert result['perplexity'] is not None
assert result['tokens_per_sec'] is not None
result